In [120]:
from typing import TypedDict, Annotated, Literal
import sqlite3
import requests

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, SystemMessage,HumanMessage,AIMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_ollama import OllamaEmbeddings
load_dotenv()
import os
from langgraph.types import interrupt,Command
from langgraph.checkpoint.memory import MemorySaver

In [121]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)

In [122]:
class Chat(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [123]:
def chat_node(chat:Chat):
    decision = interrupt({
        "type":"approval",
        "reason":"model is about to answer a question",
        "question":chat["messages"][-1].content,
        "instructions":"approve this question? yes/no"
    })
    if decision['approval']=="no":
        return {"messages":[AIMessage(content="I am not allowed to answer this question")]}
    else:
        response = llm.invoke(chat["messages"])
        return {"messages":[response]}

In [124]:
graph=StateGraph(Chat)
graph.add_node('chat_node',chat_node)
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

checkpointer=MemorySaver()
chatbot=graph.compile(checkpointer=checkpointer)

In [125]:
config={
    "configurable":{
        "thread_id":"1"
    }
}
initial_input={
    "messages":[SystemMessage(content="You are a helpful assistant"),HumanMessage(content="What is the capital of France?")]
}



In [126]:
result=chatbot.invoke(initial_input,config=config)

In [127]:
result

{'messages': [SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}, id='5a3e40d5-4d37-42f3-9f7e-c8ef6421e588'),
  HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='49263a13-26da-4a6d-ad41-27ecd89fd827')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'model is about to answer a question', 'question': 'What is the capital of France?', 'instructions': 'approve this question? yes/no'}, id='303b78375acb163505adafa19b6151e9')]}

In [128]:
message=result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'model is about to answer a question',
 'question': 'What is the capital of France?',
 'instructions': 'approve this question? yes/no'}

In [129]:
user_input=input("Enter your response (yes/no): ")


In [130]:
final_result=chatbot.invoke(
    Command(resume={"approval":user_input}),
    config=config
)

The capital of France is Paris.
